# 00 - Data Exploration (EDA)

Explore all available datasets to understand their structure, quality, and suitability for our ML features:
- **Auto-categorization**: merchant name → category
- **Anomaly detection**: unusual spending patterns
- **Spending forecast**: time series prediction
- **Savings recommendations**: user behavior analysis

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded.')

ModuleNotFoundError: No module named 'pandas'

## 1. Personal Transactions Dataset
**Primary use**: Auto-categorization training (labeled data), spending forecast, savings tips

In [ ]:
personal = pd.read_csv('../../dataset/personal_transactions.csv')
print(f'Shape: {personal.shape}')
print(f'Columns: {list(personal.columns)}')
personal.head(10)

In [ ]:
personal.info()
print()
personal.describe()

In [ ]:
# Category distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cat_counts = personal['Category'].value_counts()
cat_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Transaction Count by Category')
axes[0].set_xlabel('Count')

# Amount distribution by category
personal.groupby('Category')['Amount'].sum().sort_values().plot(
    kind='barh', ax=axes[1], color='coral'
)
axes[1].set_title('Total Amount by Category ($)')
axes[1].set_xlabel('Total Amount')

plt.tight_layout()
plt.show()

print(f'\nUnique categories: {personal["Category"].nunique()}')
print(f'Unique merchants: {personal["Description"].nunique()}')
print(f'Transaction types: {personal["Transaction Type"].value_counts().to_dict()}')

In [ ]:
# Merchant-to-category mapping analysis
merchant_cat = personal.groupby(['Description', 'Category']).size().reset_index(name='count')
merchants_multi_cat = merchant_cat.groupby('Description').size()
multi_cat = merchants_multi_cat[merchants_multi_cat > 1]
print(f'Merchants with multiple categories: {len(multi_cat)}')
if len(multi_cat) > 0:
    print(multi_cat)
print(f'\nTotal unique merchants: {personal["Description"].nunique()}')
print(f'\nMost common merchants:')
personal['Description'].value_counts().head(15)

In [ ]:
# Time series analysis
personal['Date'] = pd.to_datetime(personal['Date'], format='%m/%d/%Y')
personal.set_index('Date', inplace=False).groupby(
    pd.Grouper(key='Date', freq='M')
)['Amount'].sum().plot(kind='line', marker='o', figsize=(14, 5))
plt.title('Monthly Transaction Volume')
plt.ylabel('Total Amount ($)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Date range: {personal["Date"].min()} to {personal["Date"].max()}')
print(f'Span: {(personal["Date"].max() - personal["Date"].min()).days} days')

## 2. Credit Card Fraud Dataset
**Primary use**: Anomaly detection model training (labeled fraud/normal)

In [ ]:
# Only load first 50K rows for EDA to save memory
creditcard = pd.read_csv('../../dataset/creditcard.csv', nrows=50000)
print(f'Shape (sample): {creditcard.shape}')
print(f'Columns: {list(creditcard.columns)}')
creditcard.head()

In [ ]:
# Class distribution (fraud vs normal)
full_stats = pd.read_csv('../../dataset/creditcard.csv', usecols=['Class', 'Amount'])
class_dist = full_stats['Class'].value_counts()
print(f'Class distribution:\n{class_dist}')
print(f'\nFraud ratio: {class_dist[1] / len(full_stats) * 100:.3f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution: normal vs fraud
full_stats[full_stats['Class'] == 0]['Amount'].hist(
    bins=50, ax=axes[0], alpha=0.7, label='Normal', color='steelblue'
)
full_stats[full_stats['Class'] == 1]['Amount'].hist(
    bins=50, ax=axes[0], alpha=0.7, label='Fraud', color='red'
)
axes[0].set_title('Amount Distribution by Class')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

# Fraud amount boxplot
full_stats.boxplot(column='Amount', by='Class', ax=axes[1])
axes[1].set_title('Amount by Class')

plt.tight_layout()
plt.show()

print(f'\nNormal transactions - Amount stats:')
print(full_stats[full_stats['Class'] == 0]['Amount'].describe())
print(f'\nFraud transactions - Amount stats:')
print(full_stats[full_stats['Class'] == 1]['Amount'].describe())

del full_stats

## 3. Synthetic Personal Finance Dataset
**Primary use**: Savings recommendations, user segmentation

In [ ]:
synthetic = pd.read_csv('../../dataset/synthetic_personal_finance_dataset.csv')
print(f'Shape: {synthetic.shape}')
print(f'Columns: {list(synthetic.columns)}')
synthetic.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

synthetic['monthly_income_usd'].hist(bins=40, ax=axes[0, 0], color='green', alpha=0.7)
axes[0, 0].set_title('Monthly Income Distribution')
axes[0, 0].set_xlabel('USD')

synthetic['monthly_expenses_usd'].hist(bins=40, ax=axes[0, 1], color='red', alpha=0.7)
axes[0, 1].set_title('Monthly Expenses Distribution')
axes[0, 1].set_xlabel('USD')

synthetic['savings_to_income_ratio'].hist(bins=40, ax=axes[1, 0], color='steelblue', alpha=0.7)
axes[1, 0].set_title('Savings to Income Ratio')

synthetic['credit_score'].hist(bins=40, ax=axes[1, 1], color='purple', alpha=0.7)
axes[1, 1].set_title('Credit Score Distribution')

plt.tight_layout()
plt.show()

print('Key statistics:')
synthetic[['monthly_income_usd', 'monthly_expenses_usd', 'savings_usd', 
           'credit_score', 'debt_to_income_ratio']].describe().round(2)

## 4. Budget Reference Data
**Primary use**: Budget vs actual comparison for savings recommendations

In [ ]:
budget = pd.read_csv('../../dataset/Budget.csv')
print(f'Shape: {budget.shape}')
budget.sort_values('Budget', ascending=True).plot(
    kind='barh', x='Category', y='Budget', figsize=(10, 6),
    color='teal', legend=False
)
plt.title('Monthly Budget by Category')
plt.xlabel('Budget ($)')
plt.tight_layout()
plt.show()

print(f'Total monthly budget: ${budget["Budget"].sum()}')
budget

## 5. Summary & Next Steps

### Dataset-to-Feature Mapping
| Dataset | Rows | Auto-Cat | Anomaly | Forecast | Savings |
|---------|------|----------|---------|----------|---------|
| personal_transactions.csv | 806 | **Primary** (labeled) | Statistical | Time series | Patterns |
| creditcard.csv | 284K | - | **Primary** (fraud labels) | - | - |
| synthetic_finance.csv | 32K | - | Credit anomalies | Demographics | **Primary** |
| Budget.csv | 19 | - | Budget variance | - | Budget baseline |
| Auto-categorizer.xlsx | 116K | Volume (unlabeled) | Patterns | Volume | - |

### Category Mapping Needed
personal_transactions has 22 categories → map to our 12 app categories.
See `preprocessing.py` for the mapping.

### Proceed to:
- **Notebook 01**: Data preprocessing and category mapping
- **Notebook 02**: Auto-categorization model (Keras LSTM)